In [ ]:
import pandas as pd
import sys
sys.path.insert(0, "../src")
from features import cumulative_energy, time_since_last_heating, cumulative_bulk, cumulative_wire, previous_temperature, measurement_index

### Loading files

In [ ]:
arc = pd.read_csv("../data/raw/data_arc.csv")
bulk_additions = pd.read_csv("../data/raw/data_bulk.csv")
bulk_additions_time = pd.read_csv("../data/raw/data_bulk_time.csv")
stirring_gas = pd.read_csv("../data/raw/data_gas.csv")
temperature = pd.read_csv("../data/raw/data_temp.csv")
temperature_full = pd.read_csv("../data/raw/data_temp_FULL_with_test.csv")
wire_additions = pd.read_csv("../data/raw/data_wire.csv")
wire_additions_time = pd.read_csv("../data/raw/data_wire_time.csv")

### Data cleaning

### Instrument errors in temperature measurement

In [ ]:
error_threshold = 1500
n_before = len(temperature_full)

errors_full = temperature_full[temperature_full["Temperature"] < error_threshold]
print(f"Instrument errors found in temperature_full: {len(errors_full)}")
print(errors_full[["key", "time", "Temperature"]].to_string(index=False))

temperature_full = temperature_full[
    (temperature_full["Temperature"] >= error_threshold) | (temperature_full["Temperature"].isna())
].reset_index(drop=True)

Same removal for "temperature" dataset

In [ ]:
temperature = temperature[
    (temperature["Temperature"] >= error_threshold) | (temperature["Temperature"].isna())
].reset_index(drop=True)

print(f"\nRows removed from temperature_full: {n_before - len(temperature_full)}")
print(f"Rows remaining: {len(temperature_full)}")

### Low-gas usage

In [ ]:
low_gas = stirring_gas[stirring_gas["Gas 1"] < 1.0]
print(f"Heats with gas < 1.0 Nm³: {len(low_gas)}")
print(low_gas.to_string(index=False))

Checking if these heats have temperature measurements

In [ ]:
low_gas_keys = set(low_gas["key"])
temp_with_low_gas = temperature_full[temperature_full["key"].isin(low_gas_keys)]
print(f"\nTemperature rows for low-gas heats: {len(temp_with_low_gas)}")
print(f"Unique heats with temperature data: {temp_with_low_gas['key'].nunique()}")

Checking if low-gas heats with temperature data show consistent measured values

In [ ]:
print(temperature_full[temperature_full["key"].isin(low_gas_keys)][["key", "time", "Temperature"]].sort_values(["key", "time"]).to_string(index=False))

### Near-zero gas decision

 23 heats show gas usage below 1.0 Nm³. Of these, only 6 have temperature
 measurements — the remaining 17 have no temperature records and will be
 excluded naturally from the feature matrix since there is no target to predict.

 The 6 heats with temperature data show normal thermal behavior — temperatures
 within the expected 1500–1680°C operating window, with trajectories consistent
 with standard LF processing. This rules out process anomalies and points to
 flowmeter measurement failures as the explanation for the near-zero gas values.
 Bottom stirring is operationally mandatory: without it, temperature and
 composition gradients would persist.

 Decision: retain all 6 heats. Their thermal behavior is valid for modeling.
 The gas feature for these heats will carry the recorded (near-zero) value,
 which introduces minor noise in one feature but preserves valid temperature
 observations. Removing 6 heats to clean one feature is not a worthwhile
 trade-off.

### Timestamp conversion and merge backbone

In [ ]:
arc["Heating start"] = pd.to_datetime(arc["Heating start"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
arc["Heating end"] = pd.to_datetime(arc["Heating end"], format="%Y-%m-%d %H:%M:%S", errors="coerce")

bulk_additions_time_cols = [f"Bulk {i}" for i in range(1, 16)]
for col in bulk_additions_time_cols:
    bulk_additions_time[col] = pd.to_datetime(bulk_additions_time[col], format="%Y-%m-%d %H:%M:%S", errors="coerce")

temperature["time"] = pd.to_datetime(temperature["time"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
temperature_full["time"] = pd.to_datetime(temperature_full["time"], format="%Y-%m-%d %H:%M:%S", errors="coerce")

wire_additions_time_cols = [f"Wire {i}" for i in range(1, 10)]
for col in wire_additions_time_cols:
    wire_additions_time[col] = pd.to_datetime(wire_additions_time[col], format="%Y-%m-%d %H:%M:%S", errors="coerce")


Backbone: every temperature measurement row, sorted by melt and time

In [ ]:
backbone = temperature_full[["key", "time"]].copy()
backbone = backbone.sort_values(["key", "time"]).reset_index(drop=True)

Adding the target variable. NaN for test rows and measured value for train rows

In [ ]:
backbone = backbone.merge(
    temperature[["key", "time", "Temperature"]],
    on=["key", "time"],
    how="left"
)

backbone["is_test"] = backbone["Temperature"].isna()

Ground truth in "temperature_full" dataset for later evaluation

In [ ]:
backbone = backbone.merge(
    temperature_full[["key", "time", "Temperature"]].rename(columns={"Temperature": "true_temp"}),
    on=["key", "time"],
    how="left"
)

In [ ]:
print(f"Backbone shape: {backbone.shape}")
print(f"Train rows: {(~backbone['is_test']).sum()}")
print(f"Test rows:  {backbone['is_test'].sum()}")
print(f"Unique melts: {backbone['key'].nunique()}")
print(f"\nSample:")
display(backbone.head(10))

### Backbone structure

 The feature matrix backbone contains 15899 rows — one per temperature
 measurement across 3216 melts, after removing the 8 instrument errors
 identified in Notebook 01.

 The train/test split follows the structure defined by the original dataset:
 12999 rows where temperature was recorded (train), and 2900 rows where
 temperature was masked as NaN (test). Ground truth from
 `data_temp_FULL_with_test.csv` is stored in `true_temp` for final evaluation
 but will not be used during training or validation.

 All subsequent feature engineering operates on this backbone — each function
 takes a measurement's `key` and `time`, queries the relevant event data, and
 returns a feature value representing the physical state of the melt at that
 moment.

### Feature: cumulative energy input

In [ ]:
backbone["cumulative_energy_MJ"] = cumulative_energy(backbone, arc)

print(f"Null values: {backbone['cumulative_energy_MJ'].isna().sum()}")
print(f"\nDistribution:")
print(backbone["cumulative_energy_MJ"].describe())

In [ ]:
check_key = backbone["key"].unique()[0]
check_melt = backbone[backbone["key"] == check_key][["key", "time", "cumulative_energy_MJ"]]
check_arc = arc[arc["key"] == check_key][["Heating start", "Heating end", "Active power"]]
check_arc["duration_s"] = (check_arc["Heating end"] - check_arc["Heating start"]).dt.total_seconds()
check_arc["energy_MJ"] = check_arc["Active power"] * check_arc["duration_s"]

print("Arc sessions for this melt:")
print(check_arc.to_string(index=False))
print(f"\nMeasurements and cumulative energy:")
print(check_melt.to_string(index=False))

 ### Cumulative energy verification

 The feature contains no null values — melts missing from the arc data (2 melts
 identified in Notebook 01) correctly receive zero cumulative energy at all
 measurement points.

 The distribution spans from 0 to 8624 MJ with a median of 382 MJ, consistent
 with the per-heat energy distribution observed in the EDA. The mean (522 MJ)
 exceeds the median, reflecting the right skew from heats requiring intensive
 heating — the same pattern seen in Notebook 01.

 The spot-check on heat 1 confirms temporal causality:

 - The first measurement at 11:16 sees 499 MJ — the sum of three sessions that
   ended at 11:06, 11:10, and 11:14, all before the measurement timestamp. The
   fourth session (starting 11:18) has not yet occurred and is correctly excluded.
 - The second measurement at 11:25 sees 1105 MJ — the fourth session ended at
   11:24, so its 606 MJ contribution is now included.
 - The third, fourth, and fifth measurements (11:29–11:30) all see 1208 MJ —
   the fifth session ended at 11:28, so all five sessions are included. The
   value is constant across these three measurements because no new heating
   occurs between them.

 The monotonically non-decreasing pattern within a melt is physically correct:
 energy can only accumulate, never decrease. A measurement can see the same
 cumulative energy as its predecessor (no new heating between them) but never
 less.

### Feature: time since last heating

In [ ]:
backbone["time_since_heating_s"] = time_since_last_heating(backbone, arc)

print(f"Null values: {backbone['time_since_heating_s'].isna().sum()}")
print(f"\nDistribution (seconds):")
print(backbone["time_since_heating_s"].describe())

print("\nDistribution (minutes):")
print((backbone["time_since_heating_s"] / 60).describe())

In [ ]:
check_melt = backbone[backbone["key"] == check_key][["key", "time", "time_since_heating_s"]]
print("Measurements and time since last heating:")
print(check_melt.to_string(index=False))
print(f"\nLast heating session ended at: {arc[arc['key'] == check_key]['Heating end'].max()}")

 ### Time since heating verification

 69 measurements have NaN values — these are measurements taken before any
 heating session completed within their melt. This is physically meaningful:
 the melt has not yet received any arc energy, so "time since last heating"
 is undefined, not zero. These NaN values will be preserved in the feature
 matrix and handled natively by XGBoost, which learns the optimal routing
 for missing values at each tree split. Imputing with zero or a large constant
 would distort the physical meaning — zero would falsely signal "heating just
 ended," while a large value would falsely signal prolonged cooling.

 The distribution shows a median of 2.1 minutes with 75% of measurements
 taken within 4.4 minutes of the last heating session ending. This is
 consistent with the short-cycle LF operation observed in Notebook 01: the
 operator heats, waits briefly, then measures.

 The maximum of 179 minutes is an operational outlier — likely a heat that
 was parked in the LF station during an extended caster delay. This is not
 an error: the ladle sits idle, losing heat continuously, and the measurement
 captures the thermal state after that prolonged loss. Retaining it gives
 the model exposure to extreme cooling scenarios.

 The spot-check on heat 1 confirms the calculation:

 - First measurement at 11:16:18 shows 102 seconds — the third heating session
   ended at 11:14:36, a gap of 1 minute 42 seconds.
 - Second measurement at 11:25:53 shows 94 seconds — the fourth session ended
   at 11:24:19.
 - Third measurement at 11:29:11 shows 34 seconds — the fifth session ended
   at 11:28:37.
 - Fourth and fifth measurements at 11:30:01 and 11:30:39 show 84 and 122
   seconds respectively — both reference the same fifth session, with the gap
   growing as time passes without new heating.

### Feature: bulk additions

In [ ]:
bulk_features = cumulative_bulk(backbone, bulk_additions, bulk_additions_time)
backbone = pd.concat([backbone, bulk_features], axis=1)

print(f"New columns: {list(bulk_features.columns)}")
print(f"Null values:\n{bulk_features.isna().sum()}")
print(f"\nTotal bulk cumulative — distribution:")
print(backbone["total_bulk_cum"].describe())

In [ ]:
sample_bulk_key = None
for key in backbone["key"].unique()[:100]:
    bt_row = bulk_additions_time[bulk_additions_time["key"] == key]
    if len(bt_row) == 0:
        continue
    timestamps = bt_row[[f"Bulk {i}" for i in range(1, 16)]].values[0]
    valid_ts = [pd.Timestamp(t) for t in timestamps if pd.notna(t)]
    if len(set(valid_ts)) >= 2:
        sample_bulk_key = key
        break

print(f"Spot-check heat: {sample_bulk_key}")

bt_row = bulk_additions_time[bulk_additions_time["key"] == sample_bulk_key]
b_row = bulk_additions[bulk_additions["key"] == sample_bulk_key]
for i in range(1, 16):
    ts = bt_row[f"Bulk {i}"].values[0]
    mass = b_row[f"Bulk {i}"].values[0]
    if pd.notna(ts):
        print(f"  Bulk {i:2d}: {ts}  —  {mass:.1f} kg")

print(f"\nMeasurements:")
check = backbone[backbone["key"] == sample_bulk_key][
    ["key", "time", "total_bulk_cum"] + [f"Bulk_{i}_cum" for i in range(1, 16)]
]
print(check.to_string(index=False))

 ### Cumulative bulk verification

 All 16 bulk columns (15 individual materials + total) contain zero null values.
 Melts absent from the bulk data (87 identified in Notebook 01) correctly receive
 zero across all columns — consistent with the decision to treat missing bulk
 records as genuine absence of additions.

 The total cumulative bulk distribution has a median of 499 kg and spans from
 0 to 3235 kg, matching the per-heat bulk totals observed in the EDA. The mean
 (459 kg) sits below the median because many measurements occur early in the
 heat before all additions have been made — they see partial cumulative totals
 rather than the full heat total.

 The spot-check on heat 1 confirms both temporal causality and per-material
 tracking:

 - Bulks 12, 14, and 15 were added simultaneously at 11:03:52. The first
   measurement at 11:16:18 correctly sees all three (206 + 150 + 154 = 510 kg).
 - Bulk 4 was added at 11:21:30. The first measurement at 11:16 does not see it
   (total remains 510 kg), but the second measurement at 11:25 does (total jumps
   to 553 kg). This is the temporal causality guarantee working correctly.
 - The remaining three measurements all see 553 kg — no further additions occur,
   so the cumulative total is flat.

 Retaining individual material columns rather than only the total preserves
 information that may matter: different materials have different enthalpies of
 dissolution, and the model may learn that 200 kg of one material has a different
 thermal impact than 200 kg of another.

### Feature: wire additions

In [ ]:
wire_features = cumulative_wire(backbone, wire_additions, wire_additions_time)
backbone = pd.concat([backbone, wire_features], axis=1)

print(f"New columns: {list(wire_features.columns)}")
print(f"Null values:\n{wire_features.isna().sum()}")
print(f"\nTotal wire cumulative — distribution:")
print(backbone["total_wire_cum"].describe())

In [ ]:
sample_wire_key = None
for key in backbone["key"].unique()[:100]:
    wt_row = wire_additions_time[wire_additions_time["key"] == key]
    if len(wt_row) == 0:
        continue
    timestamps = wt_row[[f"Wire {i}" for i in range(1, 10)]].values[0]
    valid_ts = [pd.Timestamp(t) for t in timestamps if pd.notna(t)]
    if len(valid_ts) >= 1:
        sample_wire_key = key
        break

print(f"Spot-check heat: {sample_wire_key}")

wt_row = wire_additions_time[wire_additions_time["key"] == sample_wire_key]
w_row = wire_additions[wire_additions["key"] == sample_wire_key]
for i in range(1, 10):
    ts = wt_row[f"Wire {i}"].values[0]
    mass = w_row[f"Wire {i}"].values[0]
    if pd.notna(ts):
        print(f"  Wire {i}: {ts}  —  {mass:.1f} kg")

print(f"\nMeasurements:")
check = backbone[backbone["key"] == sample_wire_key][
    ["key", "time", "total_wire_cum"] + [f"Wire_{i}_cum" for i in range(1, 10)]
]
print(check.to_string(index=False))

 ### Cumulative wire verification

 All 10 wire columns (9 individual materials + total) contain zero null values.
 The 135 melts without wire records correctly receive zero across all columns —
 consistent with the Notebook 01 finding that these likely represent steel grades
 not requiring wire treatment.

 The total cumulative wire distribution has a median of 93 kg — roughly one-fifth
 of the bulk median (499 kg), confirming that wire additions are a secondary
 thermal sink. The 25th percentile is exactly zero, reflecting the substantial
 fraction of measurements where no wire has been injected yet (or no wire is used
 at all for that heat). The maximum of 664 kg is unusually high for wire and
 likely represents a heat requiring multiple wire types or repeated injection.

 The spot-check on heat 1 confirms temporal causality: Wire 1 was injected at
 11:11:41 with 60.1 kg, and all five measurements (starting at 11:16:18) correctly
 see the full amount since they all occur after the injection timestamp. No other
 wires were used in this heat, so columns Wire_2 through Wire_9 remain zero
 throughout — consistent with the EDA finding that Wire 1 (likely CaSi) is the
 dominant wire material.

### Feature: gas usage

In [ ]:
backbone = backbone.merge(
    stirring_gas[["key", "Gas 1"]].rename(columns={"Gas 1": "gas_total"}),
    on="key",
    how="left"
)

print(f"Null values in gas_total: {backbone['gas_total'].isna().sum()}")
print(f"\nDistribution:")
print(backbone["gas_total"].describe())

In [ ]:
null_gas_keys = backbone[backbone["gas_total"].isna()]["key"].unique()
print(f"Keys with null gas_total: {null_gas_keys}")

In [ ]:
print(backbone[backbone["key"].isin([193, 259])][["key", "time", "Temperature", "true_temp"]].to_string(index=False))

 10 rows across 2 heats (193 and 259) have no gas record. Both heats show
 normal temperature behavior — readings within the typical 1577–1595°C operating
 window with no anomalies — confirming these are registration failures in the gas
 file, not process anomalies. Bottom stirring is operationally mandatory: without
 it, the heat cannot be treated in LF.

 Decision: impute `gas_total = 0` for these 10 rows. This is a known limitation —
 zero does not mean no purging occurred, it means the gas flowmeter record is
 missing. With only 10 affected rows out of 15899, the impact on model learning
 is negligible.

### Feature: previous temperature

In [ ]:
backbone["prev_temp"] = previous_temperature(backbone)

print(f"Null values in prev_temp: {backbone['prev_temp'].isna().sum()}")
print(f"\nDistribution:")
print(backbone["prev_temp"].describe())

In [ ]:
mixed_key = None
for key, group in backbone.groupby("key"):
    if len(group) >= 2:
        first_is_train = not group.iloc[0]["is_test"]
        second_is_test = group.iloc[1]["is_test"]
        if first_is_train and second_is_test:
            mixed_key = key
            break

print(f"Spot-check heat: {mixed_key}")
check = backbone[backbone["key"] == mixed_key][["key", "time", "Temperature", "is_test", "prev_temp"]]
print(check.to_string(index=False))

 5377 rows have NaN in `prev_temp` — substantially more than the 3216 melts
 in the dataset. The excess comes from test rows: when a test row follows another
 test row, the preceding `Temperature` is NaN (since test labels are masked), so
 `prev_temp` propagates as NaN forward through consecutive test rows.

 The spot-check on heat 2500 illustrates this clearly:

 - Measurement 1 (train): `prev_temp = NaN` — correct, no prior measurement exists.
 - Measurement 2 (test): `prev_temp = 1539.0` — correct, inherits from the train
   row above.
 - Measurement 3 (test): `prev_temp = NaN` — correct, the preceding row is a test
   row whose `Temperature` is NaN. Using `true_temp` here would leak ground truth
   into features.
 - Measurement 4 (test): `prev_temp = NaN` — same logic, NaN propagates forward.

 This is the intended behavior. The model sees `prev_temp` only when it was
 genuinely measured and recorded — never from masked test labels. XGBoost handles
 NaN natively, routing missing values to the optimal branch at each split. The
 model will learn two distinct regimes: predictions anchored by a known prior
 temperature, and predictions without that anchor — relying more heavily on
 cumulative energy, additions, and measurement index.

### Feature: measurement index

In [ ]:
backbone["measurement_index"] = measurement_index(backbone)

print(f"Distribution:")
print(backbone["measurement_index"].describe())
print(f"\nValue counts (top 10):")
print(backbone["measurement_index"].value_counts().sort_index().head(10))

In [ ]:
check = backbone[backbone["key"] == check_key][
    ["key", "time", "measurement_index", "prev_temp", "gas_total"]
]
print(check.to_string(index=False))

 The distribution confirms the structure observed in Notebook 01: most heats have
 3–5 measurements (median 3, mean 3.3), with a long tail extending to 16. The
 steep drop-off — 3216 heats have a first measurement, but only 72 reach a tenth
 — reflects the variable complexity of LF treatments. Simple heats need few
 cycles, while complex heats requiring extensive chemistry correction or
 temperature recovery undergo many more heating-measurement iterations.

 As a feature, measurement index captures process stage: a measurement at index 1
 occurs before significant heating has accumulated, while index 5 or higher
 indicates a heat deep into its treatment cycle, with substantial cumulative
 energy and additions already applied.

### Feature matrix assembly and export

Imputing missing gas values

In [ ]:
backbone["gas_total"] = backbone["gas_total"].fillna(0)
print(f"gas_total nulls after imputation: {backbone['gas_total'].isna().sum()}")

Defining feature columns and target

In [ ]:
feature_cols = (
    ["cumulative_energy_MJ", "time_since_heating_s"]
    + [f"Bulk_{i}_cum" for i in range(1, 16)]
    + ["total_bulk_cum"]
    + [f"Wire_{i}_cum" for i in range(1, 10)]
    + ["total_wire_cum"]
    + ["gas_total", "prev_temp", "measurement_index"]
)

metadata_cols = ["key", "time", "Temperature", "true_temp", "is_test"]

print(f"Feature columns: {len(feature_cols)}")
print(f"Metadata columns: {len(metadata_cols)}")
print(f"Total columns: {len(feature_cols) + len(metadata_cols)}")
print(f"\nFeatures:\n{feature_cols}")

Checking the final shape

In [ ]:
output = backbone[metadata_cols + feature_cols].copy()
print(f"\nFinal feature matrix shape: {output.shape}")
print(f"Train rows: {(~output['is_test']).sum()}")
print(f"Test rows:  {output['is_test'].sum()}")

Null summary across all features

In [ ]:
null_summary = output[feature_cols].isnull().sum()
print("Null values per feature:")
print(null_summary[null_summary > 0])
print(f"\nFeatures with zero nulls: {(null_summary == 0).sum()} / {len(feature_cols)}")

Temporal causality validation

In [ ]:
cumulative_cols = (
    ["cumulative_energy_MJ", "total_bulk_cum", "total_wire_cum"]
)

violations = 0
for key, group in output.groupby("key"):
    for col in cumulative_cols:
        values = group[col].values
        if any(values[i] > values[i + 1] for i in range(len(values) - 1)):
            print(f"VIOLATION: melt {key}, column {col}")
            violations += 1

print(f"\nMonotonicity violations: {violations}")

Saving to "processed"

In [ ]:
output.to_csv("../data/processed/features.csv", index=False)
print(f"Shape: {output.shape}")


  Feature matrix summary

 The final feature matrix contains 15,899 rows × 36 columns: 5 metadata columns
 (`key`, `time`, `Temperature`, `true_temp`, `is_test`) and 31 feature columns.

 **Feature inventory:**

 | Feature family | Columns | Nulls | Physical meaning |
 |---|---|---|---|
 | Cumulative energy | 1 | 0 | Total MJ delivered by arc heating before measurement |
 | Time since heating | 1 | 69 | Seconds since last heating session ended — proxy for heat loss |
 | Cumulative bulk | 15 + 1 total | 0 | kg of each bulk material added before measurement |
 | Cumulative wire | 9 + 1 total | 0 | kg of each wire material added before measurement |
 | Gas total | 1 | 0 | Total purging gas for the heat (heat-level, no temporal resolution) |
 | Previous temperature | 1 | 5,377 | Last measured temperature within the melt |
 | Measurement index | 1 | 0 | Ordinal position within the melt (1-indexed) |

 Null values exist in two features by design:

 - `time_since_heating_s`: 69 NaN values — measurements taken before any heating
   session completed. The concept of "time since heating" is undefined, not zero.
 - `prev_temp`: 5,377 NaN values — first measurements in each melt (3,216) plus
   test rows following other test rows where the prior temperature is masked.

 Both are preserved as NaN for XGBoost's native missing-value handling — imputing
 would distort the physical meaning in both cases.

 Temporal causality was verified across all melts: cumulative energy, bulk, and
 wire features are monotonically non-decreasing within every melt, confirming
 that no future information leaks into any measurement's features.

 The matrix is saved to `data/processed/features.csv` for use in Notebooks 03
 and 04.